# Code Correctness Testing (Native Parsing)
This script extracts the `sft_code_solution` OR-Tools python code from the training dataset and executes it natively. 


In [1]:
import json
import subprocess
import sys
import os

# Set file path
data_path = r"/home/nickolaslow/trip-plan/p2_specialized-slm-development-pipeline/ro3_supervised-fine-tuning/data/trip_plannning_train_sft_336.json"

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples from dataset.")

def get_ground_truth(cities_str, durations_str):
    cities = cities_str.split("**")
    durations = [int(d) for d in durations_str.split("**")]
    
    gt = []
    current_start = 1
    for c, d in zip(cities, durations):
        end = current_start + d - 1
        gt.append({
            "city": c,
            "start": current_start,
            "end": end
        })
        current_start = end
    return gt


Loaded 336 examples from dataset.


In [2]:
def test_example(example_id, dataset=None):
    if dataset is None:
        dataset = data

    key = str(example_id)
    if not key.startswith("trip_planning_example_"):
        key = f"trip_planning_example_{key}"
        
    if key not in dataset:
        print(f"Error: {key} not found in the dataset.")
        return
        
    example = dataset[key]
    code_str = example.get("sft_code_solution", "")
    
    # Remove markdown tags
    if code_str.startswith("```python"):
        code_str = code_str[len("```python"):].strip()
    if code_str.endswith("```"):
        code_str = code_str[:-3].strip()
        
    # Save the NATIVE code string exactly as provided in the dataset!
    temp_file = f"temp_solve_{key}.py"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(code_str)
        
    model_solutions = []
    try:
        result = subprocess.run([sys.executable, temp_file], capture_output=True, text=True, timeout=15)
        output_str = result.stdout.strip()
        
        # Extract last line with JSON
        output_lines = [line.strip() for line in output_str.split('\n') if line.strip()]
        last_line = output_lines[-1] if output_lines else ""
        
        try:
            parsed = json.loads(last_line)
            
            # Gracefully handle multiple output formats
            if isinstance(parsed, dict) and "solutions" in parsed:
                model_solutions = parsed["solutions"]
            elif isinstance(parsed, list):
                # Output is a list of lists (SolutionCollector format) -> [[{...}], [{...}]]
                if len(parsed) > 0 and isinstance(parsed[0], list):
                    model_solutions = parsed
                # Output is a list of dicts (Single standard solution format) -> [{...}, {...}]
                elif len(parsed) > 0 and isinstance(parsed[0], dict):
                    model_solutions = [parsed] # Wrap it in a list so logic remains identical
                else:
                    model_solutions = []
        except json.JSONDecodeError:
            if "No solution found" not in output_str:
                print(f"Failed to decode JSON for {key}. Last line:\n {last_line}")
            
    except subprocess.TimeoutExpired:
        print(f"Timeout for {key}.")
    except Exception as e:
        print(f"Execution Error for {key}: {e}")
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)
            
    cities_str = example.get("cities", "")
    durations_str = example.get("durations", "")
    gt_answer = get_ground_truth(cities_str, durations_str)
    
    # Normalize model solutions to gracefully support 'start_day' and 'end_day' keys
    normalized_solutions = []
    for sol in model_solutions:
        if isinstance(sol, list):
            norm_sol = []
            for item in sol:
                if isinstance(item, dict):
                    norm_item = {'city': item.get('city')}
                    norm_item['start'] = item.get('start') if item.get('start') is not None else item.get('start_day')
                    norm_item['end'] = item.get('end') if item.get('end') is not None else item.get('end_day')
                    norm_sol.append(norm_item)
            normalized_solutions.append(norm_sol)
            
    if gt_answer in normalized_solutions:
        print(f"✅ PASS: {key} correctly models the constraints!")
        if len(normalized_solutions) > 1:
            print(f"   Note: Native code generated {len(normalized_solutions)} mathematically valid sequences!")
            for i, p in enumerate(normalized_solutions):
                print(f"   Plan {i+1}: {p}")
        else:
            print(f"   Generated Plan: {normalized_solutions[0]}")
        return True
    else:
        print(f"❌ FAIL: Output constraints mismatch for {key}!")
        print(f"   Ground Truth Expected: {gt_answer}")
        if normalized_solutions:
            print(f"   Native code produced {len(normalized_solutions)} valid plan(s), but none matched the GT:")
            for i, p in enumerate(normalized_solutions):
                print(f"   Plan {i+1}: {p}")
        else:
            print("   Native code produced 0 feasible plans or failed to execute.")
        return False

# Test specifically for example 66
test_example(66)


✅ PASS: trip_planning_example_66 correctly models the constraints!
   Generated Plan: [{'city': 'Geneva', 'start': 1, 'end': 6}, {'city': 'Brussels', 'start': 6, 'end': 11}, {'city': 'Riga', 'start': 11, 'end': 12}]


True

In [3]:
def test_all(limit=None):
    correct_count = 0
    total_count = 0
    failed_execution = 0
    failed_examples = []
    
    print(f"\nTesting all examples (limit={limit})...")
    for i, (key, _) in enumerate(data.items()):
        if limit and i >= limit:
            break
            
        print(f"\n--- {key} ---")
        if test_example(key):
            correct_count += 1
        else:
            failed_execution += 1
            failed_examples.append(key)
        total_count += 1

    print(f"\n=====================")
    print(f"       RESULTS      ")
    print(f"=====================")
    print(f"Total evaluated: {total_count}")
    print(f"Correct logic: {correct_count}")
    if total_count > 0:
        print(f"Runtime Accuracy: {(correct_count / total_count) * 100:.2f} %")
    print(f"Incorrect / Failed Executions: {failed_execution}")
    
    if failed_examples:
        print("\n--- FAILED TEST CASES ---")
        for failed_key in failed_examples:
            print(f"- {failed_key}")

test_all()



Testing all examples (limit=None)...

--- trip_planning_example_66 ---
✅ PASS: trip_planning_example_66 correctly models the constraints!
   Generated Plan: [{'city': 'Geneva', 'start': 1, 'end': 6}, {'city': 'Brussels', 'start': 6, 'end': 11}, {'city': 'Riga', 'start': 11, 'end': 12}]

--- trip_planning_example_187 ---
✅ PASS: trip_planning_example_187 correctly models the constraints!
   Generated Plan: [{'city': 'Hamburg', 'start': 1, 'end': 3}, {'city': 'Venice', 'start': 3, 'end': 6}, {'city': 'Lyon', 'start': 6, 'end': 8}]

--- trip_planning_example_127 ---
✅ PASS: trip_planning_example_127 correctly models the constraints!
   Generated Plan: [{'city': 'Bucharest', 'start': 1, 'end': 2}, {'city': 'Warsaw', 'start': 2, 'end': 3}, {'city': 'Stuttgart', 'start': 3, 'end': 5}]

--- trip_planning_example_2 ---
✅ PASS: trip_planning_example_2 correctly models the constraints!
   Generated Plan: [{'city': 'Reykjavik', 'start': 1, 'end': 2}, {'city': 'Vienna', 'start': 2, 'end': 8}, {'c

In [6]:
import json
import subprocess
import sys
import os

# Set file path
data_path = r"/home/nickolaslow/trip-plan/p2_specialized-slm-development-pipeline/ro3_supervised-fine-tuning/data/trip_planning_val_sft_48.json"

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples from dataset.")

def get_ground_truth(cities_str, durations_str):
    cities = cities_str.split("**")
    durations = [int(d) for d in durations_str.split("**")]
    
    gt = []
    current_start = 1
    for c, d in zip(cities, durations):
        end = current_start + d - 1
        gt.append({
            "city": c,
            "start": current_start,
            "end": end
        })
        current_start = end
    return gt

Loaded 48 examples from dataset.


In [7]:
test_all()


Testing all examples (limit=None)...

--- trip_planning_example_58 ---
✅ PASS: trip_planning_example_58 correctly models the constraints!
   Generated Plan: [{'city': 'Stockholm', 'start': 1, 'end': 2}, {'city': 'Reykjavik', 'start': 2, 'end': 8}, {'city': 'Athens', 'start': 8, 'end': 14}]

--- trip_planning_example_152 ---
✅ PASS: trip_planning_example_152 correctly models the constraints!
   Generated Plan: [{'city': 'Riga', 'start': 1, 'end': 7}, {'city': 'Vienna', 'start': 7, 'end': 9}, {'city': 'Venice', 'start': 9, 'end': 11}]

--- trip_planning_example_17 ---
✅ PASS: trip_planning_example_17 correctly models the constraints!
   Generated Plan: [{'city': 'Copenhagen', 'start': 1, 'end': 5}, {'city': 'Vienna', 'start': 5, 'end': 8}, {'city': 'Lyon', 'start': 8, 'end': 11}]

--- trip_planning_example_49 ---
✅ PASS: trip_planning_example_49 correctly models the constraints!
   Generated Plan: [{'city': 'Split', 'start': 1, 'end': 3}, {'city': 'Milan', 'start': 3, 'end': 9}, {'city'